<a href="https://colab.research.google.com/github/Gabo190594/recommender-system-supermarket/blob/main/notebooks/06_agent_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Clonar tu repositorio desde GitHub
!git clone https://github.com/Gabo190594/recommender-system-supermarket.git

# Entrar a la carpeta del proyecto
%cd recommender-system-supermarket

Cloning into 'recommender-system-supermarket'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 82 (delta 33), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 773.20 KiB | 3.73 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/content/recommender-system-supermarket


In [3]:
from IPython.display import Markdown, display

display(Markdown("""
# 06. Simulación de agente inteligente

## Objetivo
Representar el recomendador como un agente que percibe el estado del usuario y toma decisiones.

## ¿Qué hace este notebook?
1. Define un módulo de percepción para resumir el comportamiento del usuario.
2. Define un sistema de recompensa para medir afinidad promedio.
3. Implementa un agente que:
   - observa el estado del usuario,
   - identifica patrones básicos,
   - recomienda productos acordes a ese perfil.
4. Ejecuta una simulación con un usuario de ejemplo.

## ¿Por qué usar un agente?
Este enfoque permite mostrar que el sistema no solo calcula similitudes, sino que
también puede modelarse como una entidad que percibe información y actúa en consecuencia.

## Resultado esperado
Visualizar cómo un agente simple puede generar recomendaciones personalizadas.
"""))


# 06. Simulación de agente inteligente

## Objetivo
Representar el recomendador como un agente que percibe el estado del usuario y toma decisiones.

## ¿Qué hace este notebook?
1. Define un módulo de percepción para resumir el comportamiento del usuario.
2. Define un sistema de recompensa para medir afinidad promedio.
3. Implementa un agente que:
   - observa el estado del usuario,
   - identifica patrones básicos,
   - recomienda productos acordes a ese perfil.
4. Ejecuta una simulación con un usuario de ejemplo.

## ¿Por qué usar un agente?
Este enfoque permite mostrar que el sistema no solo calcula similitudes, sino que
también puede modelarse como una entidad que percibe información y actúa en consecuencia.

## Resultado esperado
Visualizar cómo un agente simple puede generar recomendaciones personalizadas.


In [4]:
import pandas as pd
import numpy as np
from IPython.display import Markdown, display

# =========================
# CARGA
# =========================
df = pd.read_csv("data/processed/reward_data.csv")
products = pd.read_csv("data/raw/products.csv")

# =========================
# MÓDULO DE PERCEPCIÓN
# =========================
class PerceptionModule:
    def __init__(self, data):
        self.data = data

    def get_user_state(self, user_id):
        user_data = self.data[self.data["user_id"] == user_id]

        if user_data.empty:
            return None

        favorite_category = user_data["category"].mode().iloc[0]
        avg_reward = user_data["reward_final"].mean()
        num_interactions = len(user_data)

        return {
            "user_id": user_id,
            "favorite_category": favorite_category,
            "avg_reward": avg_reward,
            "num_interactions": num_interactions
        }

# =========================
# SISTEMA DE RECOMPENSA
# =========================
class RewardSystem:
    def __init__(self, data):
        self.data = data

    def get_user_reward(self, user_id):
        user_data = self.data[self.data["user_id"] == user_id]
        if user_data.empty:
            return 0
        return user_data["reward_final"].mean()

# =========================
# AGENTE
# =========================
class IntelligentAgent:
    def __init__(self, data):
        self.data = data
        self.perception = PerceptionModule(data)
        self.reward_system = RewardSystem(data)

    def recommend(self, user_id, top_n=5):
        state = self.perception.get_user_state(user_id)

        if state is None:
            return pd.DataFrame()

        favorite_category = state["favorite_category"]

        user_seen = set(self.data[self.data["user_id"] == user_id]["product_id"].unique())

        candidates = self.data[
            (self.data["category"] == favorite_category) &
            (~self.data["product_id"].isin(user_seen))
        ]

        recs = (
            candidates.groupby(["product_id", "category"])["reward_final"]
            .mean()
            .reset_index()
            .sort_values("reward_final", ascending=False)
            .head(top_n)
        )

        return recs.merge(products, on=["product_id", "category"], how="left")

# =========================
# SIMULACIÓN
# =========================
agent = IntelligentAgent(df)

sample_user = df["user_id"].iloc[0]

state = agent.perception.get_user_state(sample_user)
display(Markdown("## 👀 Estado percibido del usuario"))
print(state)

display(Markdown("## 🎯 Reward promedio del usuario"))
print(agent.reward_system.get_user_reward(sample_user))

display(Markdown("## 🤖 Recomendaciones del agente"))
display(agent.recommend(sample_user, top_n=10))

## 👀 Estado percibido del usuario

{'user_id': np.int64(320), 'favorite_category': 'limpieza', 'avg_reward': np.float64(0.35714285714285715), 'num_interactions': 35}


## 🎯 Reward promedio del usuario

0.35714285714285715


## 🤖 Recomendaciones del agente

,product_id,category,reward_final,price
0,514,limpieza,0.471429,26.42
1,117,limpieza,0.461905,37.90
2,402,limpieza,0.460000,6.65
3,461,limpieza,0.455556,87.97
4,165,limpieza,0.452000,78.20
5,3,limpieza,0.447368,70.85
6,622,limpieza,0.446875,79.75
7,365,limpieza,0.442857,45.09
8,7,limpieza,0.440000,91.63
9,61,limpieza,0.440000,5.17
